# Notebook 8: Optimization Algorithms

Now that we have the gradients (we know which direction to move the weights), we need a strategy to actually update them. This strategy is called an **Optimizer**.

**Learning Methodology:**
**Concept $\rightarrow$ Mathematical Intuition $\rightarrow$ Pros/Cons $\rightarrow$ Implementation $\rightarrow$ Comparison**

## 1. Gradient Descent Basics

**Explanation:**
The weight update rule is simple:
$$w_{new} = w_{old} - (\text{Learning Rate} \cdot \text{Gradient})$$

**Learning Rate ($\eta$):**
The size of the step we take.
- **Too Large:** The model "overshoots" the minimum and may never converge (divergence).
- **Too Small:** The model takes forever to learn and may get stuck in a local minimum.

**Analogy:**
Walking down a mountain in the fog. The gradient is the slope under your feet, and the learning rate is the length of your step.

## 2. Types of Gradient Descent

**A. Batch Gradient Descent:**
Calculates the gradient using the **entire dataset** before updating weights.
- **Pros:** Stable convergence.
- **Cons:** Extremely slow for large data.

**B. Stochastic Gradient Descent (SGD):**
Calculates the gradient and updates weights after **every single example**.
- **Pros:** Very fast, can "jump" out of local minima.
- **Cons:** Very noisy updates (the loss curve zig-zags).

**C. Mini-batch Gradient Descent:**
The industry standard. Updates weights after a small group of examples (e.g., 32 or 64).
- **Pros:** Balance between stability and speed.

## 3. Advanced Optimizers

### SGD with Momentum
**Concept:** Adds a fraction of the previous update to the current one.
**Intuition:** Like a ball rolling down a hill; it gains momentum, helping it push through small "bumps" (local minima) and flatten out oscillations.

### RMSprop
**Concept:** Adjusts the learning rate for each weight individually based on the recent magnitude of gradients.
**Intuition:** Slows down updates for weights that are changing too wildly and speeds up updates for weights that are barely moving.

### Adam (Adaptive Moment Estimation)
**Concept:** Combines **Momentum** + **RMSprop**.
**Intuition:** The "gold standard." It tracks both the average of gradients (momentum) and the average of squared gradients (RMSprop) to provide a smooth, adaptive update for every parameter.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Setup a simple linear model
X = torch.randn(100, 1)
y = 3 * X + 0.5 + torch.randn(100, 1) * 0.1
model = nn.Linear(1, 1)
criterion = nn.MSELoss()

# --- EXPERIMENTING WITH OPTIMIZERS ---
def train_model(optimizer_type, lr):
    # Reset model
    m = nn.Linear(1, 1)
    if optimizer_type == 'SGD':
        opt = optim.SGD(m.parameters(), lr=lr)
    elif optimizer_type == 'Adam':
        opt = optim.Adam(m.parameters(), lr=lr)
    
    for epoch in range(100):
        opt.zero_grad()
        pred = m(X)
        loss = criterion(pred, y)
        loss.backward()
        opt.step()
    return loss.item()

# Compare SGD vs Adam
sgd_loss = train_model('SGD', 0.01)
adam_loss = train_model('Adam', 0.01)

print(f"Final Loss with SGD: {sgd_loss:.4f}")
print(f"Final Loss with Adam: {adam_loss:.4f}")

## 4. SGD vs Adam Comparison

| Feature | SGD | Adam |
| :--- | :--- | :--- |
| **Convergence Speed** | Slower | Much Faster |
| **Hyperparameters** | Only Learning Rate | LR, $\beta_1, \beta_2, \epsilon$ |
| **Stability** | High (if LR is low) | Very High |
| **Generalization** | Often better final accuracy | Faster to get "good enough" |
| **When to use** | Fine-tuning a model | Default choice for starting any project |

## 5. Learning Rate Experiment

**Observation:**
- If $\eta = 1.0$ (Too high) $\rightarrow$ The loss will explode to `NaN` or oscillate wildly.
- If $\eta = 0.00001$ (Too low) $\rightarrow$ The loss will barely move, and training will take days.
- If $\eta = 0.01$ (Balanced) $\rightarrow$ The loss decreases smoothly to a minimum.